# Sentiment Analysis Using Bag of Words and Logistic Regression

## Project Overview

Natural Language Processing (NLP) enables machines to understand, analyze, and extract meaningful information from human language. One of the most fundamental NLP tasks is **Sentiment Analysis**, which aims to determine whether a piece of text expresses a positive or negative opinion.

In this project, I build a complete sentiment analysis pipeline using classical NLP techniques and Machine Learning algorithms. The workflow covers the entire process from raw text preprocessing to feature extraction, model training, evaluation, and experiment documentation.

The primary objective of this project is not only to achieve high predictive performance but also to systematically investigate how different preprocessing techniques, feature engineering strategies, and machine learning models affect sentiment classification performance.

---

## Dataset

Dataset Link: **[IMBD Movie Reviews](https://www.kaggle.com/datasets/mwallerphunware/imbd-movie-reviews-for-binary-sentiment-analysis)**

The dataset consists of movie reviews labeled with their corresponding sentiment:

* Positive
* Negative

The reviews contain real-world natural language, making them suitable for evaluating text preprocessing techniques and machine learning models.

---

## Project Pipeline

The following stages are implemented throughout this project:

### 1. Text Preprocessing

* Contraction expansion
* Lowercasing
* Text cleaning using regular expressions
* Stopword removal with preserved negations
* Lemmatization
* Corpus construction

### 2. Feature Extraction

* Bag of Words (CountVectorizer)
* N-gram generation
* Vocabulary analysis
* Feature selection using `max_features`
* Rare-word filtering using `min_df`

### 3. Model Training

Different machine learning algorithms are evaluated and compared, including:

* Gaussian Naive Bayes
* Multinomial Naive Bayes
* Bernoulli Naive Bayes
* Logistic Regression
* Support Vector Machines (SVM)
* Decision Trees
* Random Forests

### 4. Model Evaluation

Performance is assessed using multiple metrics:

* Accuracy
* Precision
* Recall
* F1-Score
* ROC-AUC Score
* Confusion Matrix
* Cross-Validation Mean Accuracy
* Cross-Validation Standard Deviation

---

## Experimental Approach

Rather than training a single model, this notebook follows an experimentation-driven methodology.

For each model, multiple configurations are tested, including different values for:

* `max_features`
* `min_df`
* N-gram ranges
* Preprocessing strategies

The goal is to identify the most effective configuration while understanding the trade-offs between model complexity, computational cost, and predictive performance.

---

## Key Learning Objectives

Through this project, I aim to:

* Develop a deeper understanding of NLP preprocessing techniques.
* Compare the behavior of different machine learning algorithms on text data.
* Analyze how feature engineering impacts classification performance.
* Build reproducible NLP pipelines suitable for real-world applications.
* Establish strong baselines before moving toward advanced embedding and transformer-based approaches.

---

**Author:** Hazem Mohamed

**Role:** AI Engineer | Machine Learning Engineer | NLP Engineer

**Repository:** [NLP Experimentation Lab](https://github.com/Hazem1695/NLP-Experimentation-Lab)


# **Importing the Libraries**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# **Data preprocessing**

## Data Cleaning Check Template
This template is designed to quickly assess the quality of any dataset before building machine learning models or performing analysis.

It provides a structured overview of the dataset by checking for common data issues such as:

- Missing values

- Duplicate rows

- Incorrect data types

- Outliers

- Distribution of numerical features

- Categorical feature consistency

**What This Template Does**

- Displays basic dataset information (shape, data types)

- Identifies missing values and duplicates

- Summarizes numerical and categorical features

- Detects potential outliers using the IQR method

- Highlights columns with low unique values for quick inspection

How to Use

1. Load your dataset using Pandas  

2. Call the function:

In [2]:
def data_quality_report(df):

    print("DATA QUALITY REPORT")
    
    # Print a separator line for better readability
    
    print("=" * 50)
    print("BASIC INFO")
    print("=" * 50)
    
    # Show general information about the dataset (columns, data types, non-null values)
    print(df.info())
    
    # Show number of rows and columns
    print("\n" + "=" * 50)
    print("SHAPE OF DATA")
    print("=" * 50)
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    
    # Check for missing (null) values in each column
    print("\n" + "=" * 50)
    print("MISSING VALUES")
    print("=" * 50)
    missing = df.isnull().sum()
    
    # Display only columns that have missing values
    print(missing[missing > 0])
    
    # Check for duplicate rows
    print("\n" + "=" * 50)
    print("DUPLICATES")
    print("=" * 50)
    print(f"Duplicate rows: {df.duplicated().sum()}")
    
    # Display data types of each column
    print("\n" + "=" * 50)
    print("DATA TYPES")
    print("=" * 50)
    print(df.dtypes)
    
    # Summary statistics for numerical columns (mean, std, min, max, etc.)
    print("\n" + "=" * 50)
    print("NUMERICAL SUMMARY")
    print("=" * 50)
    print(df.describe())
    
    # Summary for categorical (object) columns
    print("\n" + "=" * 50)
    print("CATEGORICAL SUMMARY")
    print("=" * 50)
    print(df.describe(include=['object']))
    
    # Show unique values for columns with low number of distinct values
    # Useful for detecting categories, errors, or inconsistencies
    print("\n" + "=" * 50)
    print("UNIQUE VALUES (LOW CARDINALITY)")
    print("=" * 50)
    for col in df.columns:
        if df[col].nunique() < 10:  # Only show columns with few unique values
            print(f"{col}: {df[col].unique()}")
            
    # correlation
    print("\n" + "=" * 50)
    print("CORRELATION MATRIX")
    print("=" * 50)
    print(df.corr(numeric_only=True))
    
    # Detect outliers using the IQR (Interquartile Range) method
    print("\n" + "=" * 50)
    print("OUTLIERS CHECK (IQR METHOD)")
    print("=" * 50)
    
    # Loop through only numerical columns
    for col in df.select_dtypes(include=np.number).columns:
        Q1 = df[col].quantile(0.25)  # 25th percentile
        Q3 = df[col].quantile(0.75)  # 75th percentile
        IQR = Q3 - Q1  # Interquartile range
        
        # Count rows that fall outside the normal range
        outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
        print(f"{col}: {len(outliers)} outliers")

## **Load dataset**
Apply Data Cleaning Check Template

In [ ]:
dataset = pd.read_csv('MovieReviewTrainingDatabase.csv')
data_quality_report(dataset)

DATA QUALITY REPORT
BASIC INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  25000 non-null  object
 1   review     25000 non-null  object
dtypes: object(2)
memory usage: 390.8+ KB
None

SHAPE OF DATA
Rows: 25000, Columns: 2

MISSING VALUES
Series([], dtype: int64)

DUPLICATES
Duplicate rows: 96

DATA TYPES
sentiment    object
review       object
dtype: object

NUMERICAL SUMMARY
       sentiment                                             review
count      25000                                              25000
unique         2                                              24904
top     Positive  You do realize that you've been watching the E...
freq       12500                                                  3

CATEGORICAL SUMMARY
       sentiment                                             review
count      25000                 

## Duplicate Data Detection

In [4]:
duplicates = dataset[dataset.duplicated(subset=['review'], keep=False)]
duplicates.sort_values('review')

,sentiment,review
21186,Negative,"Back in his youth, the old man had wanted to..."
21877,Negative,"Back in his youth, the old man had wanted to..."
14734,Negative,'Dead Letter Office' is a low-budget film abou...
5519,Negative,'Dead Letter Office' is a low-budget film abou...
7011,Positive,".......Playing Kaddiddlehopper, Col San Fernan..."
...,...,...
2685,Negative,"in this movie, joe pesci slams dunks a basketb..."
22244,Positive,it's amazing that so many people that i know h...
14767,Positive,it's amazing that so many people that i know h...
12462,Negative,this movie begins with an ordinary funeral... ...


## Quantifying Duplicate Review Frequencies

In [5]:
review_counts = dataset['review'].value_counts()
print("Reviews appearing more than once:")
print((review_counts > 1).sum())
print("\nMaximum repetitions:")
print(review_counts.max())

Reviews appearing more than once:
92

Maximum repetitions:
3


## Removing Duplicate Reviews & Resetting Index
> **Note:** This cell drops the repeated rows we identified in the previous steps and cleanly resets the row indices for model training

In [6]:
print("Before:", len(dataset))
dataset = dataset.drop_duplicates()
print("After:", len(dataset))
dataset = dataset.reset_index(drop=True)

Before: 25000
After: 24904


## Library Installation
> **Note:** The `contractions` library is required to automatically expand shortcuts like *don't* to *do not* and *I'm* to *I am* during preprocessing.

In [7]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 7.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.4 MB/s eta 0:00:00


## **Cleaning the texts**

In [8]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
import re
import contractions
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

base_stopwords = set(stopwords.words('english'))
negation_words = {'not', 'no', 'never'}
all_stopwords = base_stopwords - negation_words

wnl = WordNetLemmatizer()

corpus = []

for i in range(0, len(dataset)):
    review = dataset['review'][i]
    # Fix contractions
    review = contractions.fix(review)
    # Lowercase
    review = review.lower()
    
    review = re.sub(r'[^a-zA-Z\s]', ' ', review)
    # Split
    words = review.split()
    # Chained Lemmatization (Handles both Verbs 'v' and Nouns 'n')
    review = [wnl.lemmatize(wnl.lemmatize(word, pos='v'), pos='n') for word in words if word not in all_stopwords]
    review = ' '.join(review) 
    corpus.append(review)

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


## Preprocessing Verification
> **Note:** Pulling the first two rows directly as a memory array to confirm that our lowercasing, stopword stripping, and lemmatization pipeline worked correctly before feeding it into the vectorizer.

In [9]:
# Pull the data directly as a fast memory array
raw_samples = dataset['review'].head(2).values

for i in range(2):
    print(f"=== REVIEW #{i+1} ===")
    print(f"RAW:     {raw_samples[i]}\n") 
    print(f"CLEANED: {corpus[i]}")
    print("-" * 50)

=== REVIEW #1 ===
RAW:     With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.  Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.  The actual feature film bit when it final

# **Encoding Categorical data Using Label Encoding**

In [10]:
from sklearn.preprocessing import LabelEncoder
y = dataset.iloc[:, 0].values
le = LabelEncoder()
y = le.fit_transform(y)

In [11]:
print(y)

[1 1 0 ... 0 0 1]


# **Splitting the dataset into the Training set and Test set**

In [12]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(corpus, y, test_size = 0.20, random_state = 0)

# Class Balance Check
> **Note:** Using NumPy to verify if our dataset is perfectly balanced between positive and negative reviews before splitting it into training and testing sets.

In [13]:
import numpy as np
# This returns the unique classes and how many times they appear
classes, counts = np.unique(y, return_counts=True)
for c, count in zip(classes, counts):
    print(f"Class {c} contains {count}")

Class 0 contains 12432
Class 1 contains 12472


# **Creating the Bag of Word model**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
# Add ngram_range=(1, 2) so it automatically catches phrases like "not good" or "no clue"
cv = CountVectorizer(max_features=30000, min_df=2, ngram_range=(1, 2))
X_train = cv.fit_transform(X_train)
X_test = cv.transform(X_test)

In [15]:
print(X_train.shape)        # (n_samples, n_features)
print(X_train.shape[0])     # number of training samples
print(X_train.shape[1])     # number of features (vocabulary size)

(19923, 30000)
19923
30000


In [16]:
print(X_test.shape)        # (n_samples, n_features)
print(X_test.shape[0])     # number of training samples
print(X_test.shape[1])     # number of features (vocabulary size)

(4981, 30000)
4981
30000


# **Training the Logistic Regression model on the Training set**

In [17]:
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression(max_iter=1000, random_state = 0)
classifier.fit(X_train,y_train)

LogisticRegression(max_iter=1000, random_state=0)

# **Predicting the Test set results**

In [18]:
y_pred = classifier.predict(X_test)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[1 1]
 [1 1]
 [0 0]
 ...
 [0 0]
 [0 0]
 [0 0]]


# **Evaluating the Model Performance**

In [19]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_pred))


accuracies = cross_val_score(estimator=classifier, X=X_train, y=y_train, cv=3)

print("\nMean Accuracy:")
print(accuracies.mean())

print("\nStandard Deviation:")
print(accuracies.std())

Confusion Matrix:
[[2194  326]
 [ 233 2228]]

Accuracy Score:
0.8877735394499097

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.87      0.89      2520
           1       0.87      0.91      0.89      2461

    accuracy                           0.89      4981
   macro avg       0.89      0.89      0.89      4981
weighted avg       0.89      0.89      0.89      4981


ROC-AUC Score:
0.8879789800248964

Mean Accuracy:
0.8809416252572403

Standard Deviation:
0.0036911662522411733


# Logistic Regression Performance Report

## Experiment Series: CountVectorizer + Logistic Regression

---

# Overview

This experiment evaluates the performance of **Logistic Regression**, one of the strongest linear classifiers for Natural Language Processing (NLP), on a text classification task using different **CountVectorizer** configurations.

The objective was to investigate how feature engineering parameters—including vocabulary size, n-gram representation, and document frequency thresholds—affect the classification performance of Logistic Regression.

Multiple experiments were conducted while maintaining identical train/test splits and evaluation metrics to ensure a fair comparison.

---

# Experimental Configurations

The following vectorization strategies were evaluated:

| Experiment   | CountVectorizer Configuration                      | Logistic Regression |
| ------------ | -------------------------------------------------- | ------------------- |
| Experiment 1 | Default CountVectorizer                            | Default Parameters  |
| Experiment 2 | Default CountVectorizer                            | max_iter = 1000     |
| Experiment 3 | max_features = 5,000, ngram_range=(1,2)            | max_iter = 1000     |
| Experiment 4 | max_features = 15,000, ngram_range=(1,2)           | max_iter = 1000     |
| Experiment 5 | max_features = 20,000, min_df=2, ngram_range=(1,2) | max_iter = 1000     |
| Experiment 6 | max_features = 25,000, min_df=2, ngram_range=(1,2) | max_iter = 1000     |
| Experiment 7 | max_features = 30,000, min_df=2, ngram_range=(1,2) | max_iter = 1000     |

---

# Evaluation Metrics

The following metrics were used throughout all experiments:

* Accuracy
* Precision
* Recall
* F1-Score
* ROC-AUC
* Cross-Validation Mean Accuracy
* Cross-Validation Standard Deviation
* Confusion Matrix

These metrics provide a comprehensive evaluation of both predictive performance and model generalization.

---

# Results Summary

| Experiment              |   Accuracy |    ROC-AUC |    CV Mean |        Std |
| ----------------------- | ---------: | ---------: | ---------: | ---------: |
| Default CountVectorizer | **88.92%** | **0.8894** | **88.06%** |     0.0031 |
| Default + max_iter=1000 |     87.35% |     0.8738 |     86.56% | **0.0010** |
| 5K Features + Bigrams   |     85.79% |     0.8580 |     85.57% |     0.0037 |
| 15K Features + Bigrams  |     87.85% |     0.8787 |     87.44% |     0.0049 |
| 20K Features + min_df=2 |     88.26% |     0.8828 |     87.73% |     0.0034 |
| 25K Features + min_df=2 |     88.44% |     0.8846 |     87.87% |     0.0032 |
| 30K Features + min_df=2 | **88.78%** | **0.8880** | **88.09%** |     0.0037 |

---

# Best Performing Configuration

### CountVectorizer

* max_features = 30,000
* min_df = 2
* ngram_range = (1,2)

### Logistic Regression

* max_iter = 1000
* random_state = 0

---

## Performance

| Metric                    | Score      |
| ------------------------- | ---------- |
| Accuracy                  | **88.78%** |
| Precision (Class 0)       | 0.90       |
| Recall (Class 0)          | 0.87       |
| Precision (Class 1)       | 0.87       |
| Recall (Class 1)          | 0.91       |
| Macro F1                  | **0.89**   |
| ROC-AUC                   | **0.8880** |
| Cross Validation Accuracy | **88.09%** |

---

# Confusion Matrix Analysis

Best configuration:

```
               Predicted

             Negative   Positive

Actual Negative   2194      326
Actual Positive    233     2228
```

Correct classifications:

* True Negatives: **2,194**
* True Positives: **2,228**

Misclassifications:

* False Positives: **326**
* False Negatives: **233**

The confusion matrix demonstrates a balanced classifier with low error rates across both classes.

---

# Precision and Recall Analysis

The classifier maintains an excellent balance between precision and recall.

## Class 0

* Precision: **0.90**
* Recall: **0.87**

The model predicts negative samples with high confidence while maintaining strong coverage.

## Class 1

* Precision: **0.87**
* Recall: **0.91**

The higher recall indicates the classifier successfully captures the majority of positive instances, making it suitable when minimizing false negatives is important.

---

# Cross-Validation Analysis

Cross-validation confirms that Logistic Regression generalizes consistently across different data splits.

Best configuration:

* Mean Accuracy: **88.09%**
* Standard Deviation: **0.0037**

The very low standard deviation indicates excellent stability and low sensitivity to training data variations.

---

# Impact of Feature Engineering

## 1. Increasing max_iter

Increasing the optimization iterations from the default to 1000 successfully eliminated the convergence warning. However, model accuracy decreased slightly, indicating that optimization convergence alone does not guarantee improved predictive performance.

---

## 2. Using Bigrams

Introducing bigram features initially reduced performance when the vocabulary size was restricted to only 5,000 features.

This suggests that the limited feature space forced informative unigrams to compete with newly introduced bigrams.

---

## 3. Increasing Vocabulary Size

Expanding the vocabulary from:

* 5,000
* 15,000
* 20,000
* 25,000
* 30,000

produced a steady improvement in every evaluation metric.

Larger vocabularies allowed the model to capture richer semantic information and more discriminative textual patterns.

---

## 4. Applying min_df = 2

Removing extremely rare words reduced feature sparsity and eliminated noisy tokens.

This resulted in:

* better generalization,
* slightly higher ROC-AUC,
* improved cross-validation performance,
* more stable optimization.

---

# Model Strengths

* Excellent overall accuracy (≈89%)
* Balanced precision and recall
* High ROC-AUC
* Strong cross-validation performance
* Extremely stable across folds
* Fast training
* Efficient inference
* Highly interpretable coefficients
* Well suited for high-dimensional sparse text data

---

# Limitations

* Performance depends heavily on feature engineering.
* Logistic Regression cannot capture complex nonlinear relationships.
* Bag-of-Words ignores word order beyond the selected n-gram range.
* Rare contextual semantics are not represented effectively.

---

# Convergence Warning Discussion

The first experiment produced the warning:

> lbfgs failed to converge (status=1): TOTAL NO. OF ITERATIONS REACHED LIMIT.

This warning indicates that the optimizer reached the default iteration limit before full convergence.

Increasing **max_iter** to **1000** successfully resolved the warning.

Interestingly, despite achieving optimization convergence, predictive performance decreased slightly, demonstrating that the highest optimization convergence does not necessarily correspond to the highest classification accuracy.

---

# Overall Findings

Several important observations emerged from this study:

* Logistic Regression consistently achieved approximately **89% accuracy**, demonstrating strong suitability for text classification.
* Increasing the vocabulary size substantially improved performance after introducing bigram features.
* Filtering infrequent words using **min_df=2** improved model robustness.
* The classifier maintained nearly identical precision and recall across both classes, indicating balanced decision boundaries.
* Cross-validation confirmed excellent stability and reproducibility.
* Logistic Regression remains one of the strongest baseline algorithms for sparse Bag-of-Words representations due to its simplicity, efficiency, and reliable generalization.

---

# Final Conclusion

This experimental study demonstrates that **Logistic Regression remains one of the most effective classical machine learning algorithms for text classification** when combined with carefully engineered Bag-of-Words features.

Among all evaluated configurations, the combination of **CountVectorizer (30,000 features, bigrams, min_df=2)** and **Logistic Regression (max_iter=1000)** delivered the best balance between predictive accuracy, ROC-AUC, stability, and generalization.

Although more sophisticated deep learning models may achieve higher performance on large-scale datasets, Logistic Regression continues to provide an outstanding trade-off between computational efficiency, interpretability, training speed, and predictive quality, making it an excellent baseline and a highly competitive solution for many real-world NLP classification tasks.
